# 📚 shadcn/ui & TailwindCSS RAG 問答系統

本 Notebook 建立一個以 **Retrieval-Augmented Generation (RAG)** 為核心的問答系統：

| 項目 | 說明 |
|------|------|
| 語料來源 1 | [shadcn/ui](https://ui.shadcn.com/docs) — 40 頁元件文件 |
| 語料來源 2 | [TailwindCSS](https://tailwindcss.com/docs) — 46 頁工具文件 |
| Embedding | `intfloat/multilingual-e5-small` (384 維) |
| 向量資料庫 | PostgreSQL + pgvector |
| LLM | HuggingFace Transformers (`Qwen/Qwen2.5-7B-Instruct`，4-bit 量化) |
| 介面 | Gradio Chat UI |

## 架構流程
```
Web Scraper → 文字分塊 → E5 Embedding → pgvector 儲存
                                              ↓
使用者問題 → E5 Embedding → 向量搜尋 → 取回相關段落
                                              ↓
                            組合 Prompt → Qwen2.5 (HF) → 回答
```

## 啟動步驟
```bash
# 1. 啟動 PostgreSQL
docker compose up -d

# 2. 執行本 Notebook（由上到下）
#    首次執行 Cell 10 時會自動從 HuggingFace 下載模型（約 4-14GB）

# 3. Gradio UI 在 http://localhost:7860
```

In [86]:
# Cell 1: 安裝相依套件
!pip install requests beautifulsoup4 sentence-transformers psycopg2-binary gradio google-generativeai python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [87]:
# Cell 2: 匯入套件
import os
import requests
from bs4 import BeautifulSoup
import psycopg2
from sentence_transformers import SentenceTransformer
import gradio as gr
import re
import time
import google.generativeai as genai
from typing import Optional, List, Tuple

print('✅ 套件匯入完成')

✅ 套件匯入完成


In [88]:
# Cell 3: 設定參數
from dotenv import load_dotenv
load_dotenv()  # 載入 week4/.env 的環境變數

DB_CONFIG = {
    'host':     'localhost',
    'port':     5435,
    'database': 'qa_db',
    'user':     'postgres',
    'password': 'password',
}

GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')
GEMINI_MODEL   = 'gemini-3-flash-preview'

print(f'DB     → {DB_CONFIG["host"]}:{DB_CONFIG["port"]}/{DB_CONFIG["database"]}')
print(f'LLM    → Gemini ({GEMINI_MODEL})')
print(f'API Key → {"✅ 已設定" if GEMINI_API_KEY else "❌ 未找到，請確認 .env 檔案"}')

DB     → localhost:5435/qa_db
LLM    → Gemini (gemini-3-flash-preview)
API Key → ✅ 已設定


In [89]:
# Cell 4: 建立資料庫 Schema

def get_conn():
    return psycopg2.connect(**DB_CONFIG)

def setup_database(reset: bool = False):
    conn = get_conn()
    with conn.cursor() as cur:
        cur.execute('CREATE EXTENSION IF NOT EXISTS vector;')
        if reset:
            cur.execute('DROP TABLE IF EXISTS docs;')
            print('🗑  已清除舊資料')
        cur.execute('''
            CREATE TABLE IF NOT EXISTS docs (
                id          BIGINT PRIMARY KEY GENERATED BY DEFAULT AS IDENTITY,
                source      TEXT    NOT NULL,
                url         TEXT    NOT NULL,
                title       TEXT,
                chunk_index INTEGER DEFAULT 0,
                content     TEXT    NOT NULL,
                embedding   vector(384)
            );
        ''')
        conn.commit()
    conn.close()
    print('✅ 資料庫 Schema 就緒')

# 第一次執行時不 reset；若需重建語料庫，改為 reset=True
setup_database(reset=False)

✅ 資料庫 Schema 就緒


In [90]:
# Cell 5: 網頁爬蟲與文字分塊工具

def scrape_page(url: str, source: str) -> Optional[dict]:
    """抓取一個文件頁面，回傳純文字內容。"""
    headers = {
        'User-Agent': (
            'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 '
            '(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        ),
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.5',
    }
    try:
        resp = requests.get(url, headers=headers, timeout=20)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, 'html.parser')

        # 移除不需要的區塊
        for tag in soup.find_all(['nav', 'script', 'style', 'footer', 'aside', 'header']):
            tag.decompose()

        # 取頁面標題
        h1 = soup.find('h1')
        title = h1.get_text(strip=True) if h1 else url.rstrip('/').split('/')[-1]

        # 優先找 main / article 等語意標籤
        content_el = (
            soup.find('main') or
            soup.find('article') or
            soup.find(attrs={'role': 'main'}) or
            soup.find('div', class_=re.compile(r'prose|content|docs|markdown', re.I))
        )

        target = content_el if content_el else soup.find('body')
        if not target:
            return None

        text = target.get_text(separator='\n', strip=True)
        text = re.sub(r'\n{3,}', '\n\n', text)
        text = re.sub(r'[ \t]{2,}', ' ', text)
        text = text.strip()

        if len(text) < 80:
            return None

        return {'url': url, 'source': source, 'title': title, 'content': text}

    except Exception as e:
        print(f'  ⚠️  {url} → {e}')
        return None


def chunk_text(text: str, max_chars: int = 800, overlap: int = 150) -> List[str]:
    """以段落為單位切塊，並加入 overlap 維持上下文。"""
    paragraphs = [p.strip() for p in re.split(r'\n{2,}', text) if len(p.strip()) > 30]

    chunks: List[str] = []
    current = ''

    for para in paragraphs:
        if current and len(current) + len(para) > max_chars:
            chunks.append(current.strip())
            # 保留最後 overlap 字元作為下一塊的開頭
            tail = current[-overlap:] if len(current) > overlap else current
            current = tail + '\n\n' + para
        else:
            current = (current + '\n\n' + para).strip() if current else para

    if current.strip():
        chunks.append(current.strip())

    # Fallback：若無段落分隔，按字元切
    if not chunks and text:
        step = max_chars - overlap
        chunks = [text[i:i + max_chars] for i in range(0, len(text), step)]

    return [c for c in chunks if len(c) >= 50]


print('✅ 爬蟲與分塊工具定義完成')

✅ 爬蟲與分塊工具定義完成


In [91]:
# Cell 6: 定義爬取頁面清單

SHADCN_PAGES = [
    # 核心文件
    'https://ui.shadcn.com/docs',
    'https://ui.shadcn.com/docs/installation',
    'https://ui.shadcn.com/docs/cli',
    'https://ui.shadcn.com/docs/theming',
    'https://ui.shadcn.com/docs/dark-mode',
    # 元件
    'https://ui.shadcn.com/docs/components/accordion',
    'https://ui.shadcn.com/docs/components/alert',
    'https://ui.shadcn.com/docs/components/alert-dialog',
    'https://ui.shadcn.com/docs/components/avatar',
    'https://ui.shadcn.com/docs/components/badge',
    'https://ui.shadcn.com/docs/components/button',
    'https://ui.shadcn.com/docs/components/calendar',
    'https://ui.shadcn.com/docs/components/card',
    'https://ui.shadcn.com/docs/components/checkbox',
    'https://ui.shadcn.com/docs/components/command',
    'https://ui.shadcn.com/docs/components/context-menu',
    'https://ui.shadcn.com/docs/components/data-table',
    'https://ui.shadcn.com/docs/components/date-picker',
    'https://ui.shadcn.com/docs/components/dialog',
    'https://ui.shadcn.com/docs/components/dropdown-menu',
    'https://ui.shadcn.com/docs/components/form',
    'https://ui.shadcn.com/docs/components/hover-card',
    'https://ui.shadcn.com/docs/components/input',
    'https://ui.shadcn.com/docs/components/label',
    'https://ui.shadcn.com/docs/components/navigation-menu',
    'https://ui.shadcn.com/docs/components/popover',
    'https://ui.shadcn.com/docs/components/progress',
    'https://ui.shadcn.com/docs/components/radio-group',
    'https://ui.shadcn.com/docs/components/select',
    'https://ui.shadcn.com/docs/components/separator',
    'https://ui.shadcn.com/docs/components/sheet',
    'https://ui.shadcn.com/docs/components/skeleton',
    'https://ui.shadcn.com/docs/components/slider',
    'https://ui.shadcn.com/docs/components/switch',
    'https://ui.shadcn.com/docs/components/table',
    'https://ui.shadcn.com/docs/components/tabs',
    'https://ui.shadcn.com/docs/components/textarea',
    'https://ui.shadcn.com/docs/components/toast',
    'https://ui.shadcn.com/docs/components/toggle',
    'https://ui.shadcn.com/docs/components/tooltip',
]

TAILWIND_PAGES = [
    # 核心概念
    'https://tailwindcss.com/docs/installation',
    'https://tailwindcss.com/docs/utility-first',
    'https://tailwindcss.com/docs/responsive-design',
    'https://tailwindcss.com/docs/hover-focus-and-other-states',
    'https://tailwindcss.com/docs/dark-mode',
    'https://tailwindcss.com/docs/reusing-styles',
    'https://tailwindcss.com/docs/adding-custom-styles',
    'https://tailwindcss.com/docs/configuration',
    'https://tailwindcss.com/docs/theme',
    'https://tailwindcss.com/docs/preflight',
    # Layout
    'https://tailwindcss.com/docs/container',
    'https://tailwindcss.com/docs/columns',
    'https://tailwindcss.com/docs/display',
    'https://tailwindcss.com/docs/position',
    'https://tailwindcss.com/docs/overflow',
    'https://tailwindcss.com/docs/z-index',
    # Flexbox & Grid
    'https://tailwindcss.com/docs/flex',
    'https://tailwindcss.com/docs/flex-direction',
    'https://tailwindcss.com/docs/flex-wrap',
    'https://tailwindcss.com/docs/justify-content',
    'https://tailwindcss.com/docs/align-items',
    'https://tailwindcss.com/docs/gap',
    'https://tailwindcss.com/docs/grid-template-columns',
    'https://tailwindcss.com/docs/grid-column',
    # Spacing
    'https://tailwindcss.com/docs/padding',
    'https://tailwindcss.com/docs/margin',
    'https://tailwindcss.com/docs/space',
    # Sizing
    'https://tailwindcss.com/docs/width',
    'https://tailwindcss.com/docs/height',
    'https://tailwindcss.com/docs/max-width',
    # Typography
    'https://tailwindcss.com/docs/font-size',
    'https://tailwindcss.com/docs/font-weight',
    'https://tailwindcss.com/docs/text-color',
    'https://tailwindcss.com/docs/text-align',
    'https://tailwindcss.com/docs/line-height',
    # Backgrounds & Borders
    'https://tailwindcss.com/docs/background-color',
    'https://tailwindcss.com/docs/background-image',
    'https://tailwindcss.com/docs/border-radius',
    'https://tailwindcss.com/docs/border-width',
    'https://tailwindcss.com/docs/border-color',
    # Effects & Transitions
    'https://tailwindcss.com/docs/box-shadow',
    'https://tailwindcss.com/docs/opacity',
    'https://tailwindcss.com/docs/transition-property',
    'https://tailwindcss.com/docs/animation',
    'https://tailwindcss.com/docs/transform',
    'https://tailwindcss.com/docs/cursor',
]

print(f'shadcn/ui 頁面數：{len(SHADCN_PAGES)}')
print(f'TailwindCSS 頁面數：{len(TAILWIND_PAGES)}')

shadcn/ui 頁面數：40
TailwindCSS 頁面數：46


In [92]:
# Cell 7: 載入 Embedding 模型

print('載入 intfloat/multilingual-e5-small ...')
embed_model = SentenceTransformer('intfloat/multilingual-e5-small')
print('✅ Embedding 模型就緒（384 維）')


def get_embedding(text: str, is_query: bool = False) -> list:
    """產生向量，查詢加 'query: ' 前綴，文件加 'passage: ' 前綴。"""
    prefix = 'query: ' if is_query else 'passage: '
    return embed_model.encode(
        prefix + text[:512].strip(),
        normalize_embeddings=True
    ).tolist()

載入 intfloat/multilingual-e5-small ...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4579.34it/s]
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding 模型就緒（384 維）


In [93]:
# Cell 8: 爬取、分塊、向量化並存入資料庫

def scrape_and_index(pages: List[str], source: str, skip_existing: bool = True) -> int:
    """爬取頁面列表並存入向量資料庫；skip_existing=True 跳過已索引的頁面。"""
    conn = get_conn()
    total_chunks = 0

    for i, url in enumerate(pages, 1):
        slug = url.rstrip('/').split('/')[-1]

        # 檢查是否已索引
        if skip_existing:
            with conn.cursor() as cur:
                cur.execute('SELECT COUNT(*) FROM docs WHERE url = %s', (url,))
                if cur.fetchone()[0] > 0:
                    print(f'[{i:02d}/{len(pages)}] ✓ 已存在：{slug}')
                    continue

        print(f'[{i:02d}/{len(pages)}] 爬取中：{slug}', end='', flush=True)
        doc = scrape_page(url, source)

        if not doc:
            print(' ⚠️  失敗，略過')
            continue

        chunks = chunk_text(doc['content'])

        # 批次 Embedding
        raw_texts = ['passage: ' + c[:512] for c in chunks]
        embeddings = embed_model.encode(
            raw_texts, normalize_embeddings=True, batch_size=32, show_progress_bar=False
        )

        # 寫入 DB
        with conn.cursor() as cur:
            for j, (chunk, emb) in enumerate(zip(chunks, embeddings)):
                cur.execute(
                    '''
                    INSERT INTO docs (source, url, title, chunk_index, content, embedding)
                    VALUES (%s, %s, %s, %s, %s, %s)
                    ''',
                    (source, doc['url'], doc['title'], j, chunk, emb.tolist())
                )
        conn.commit()

        total_chunks += len(chunks)
        print(f' → {len(chunks)} 塊')
        time.sleep(0.4)  # 避免過度請求

    conn.close()
    return total_chunks


# ── 開始索引 ──────────────────────────────────────────────────
print('=' * 55)
print('索引 shadcn/ui 文件')
print('=' * 55)
n_shadcn = scrape_and_index(SHADCN_PAGES, 'shadcn')

print()
print('=' * 55)
print('索引 TailwindCSS 文件')
print('=' * 55)
n_tailwind = scrape_and_index(TAILWIND_PAGES, 'tailwind')

# ── 建立向量索引 ──────────────────────────────────────────────
print()
conn = get_conn()
with conn.cursor() as cur:
    cur.execute('SELECT COUNT(*) FROM docs')
    total_rows = cur.fetchone()[0]

    # IVFFlat 建議 lists ≈ sqrt(n_rows)
    lists = max(10, min(int(total_rows ** 0.5), 200))
    cur.execute('DROP INDEX IF EXISTS docs_embedding_idx')
    cur.execute(f'''
        CREATE INDEX docs_embedding_idx
        ON docs USING ivfflat (embedding vector_cosine_ops)
        WITH (lists = {lists})
    ''')
    conn.commit()

    cur.execute('SELECT source, COUNT(*) FROM docs GROUP BY source ORDER BY source')
    rows = cur.fetchall()
conn.close()

print(f'✅ 向量索引建立完成 (lists={lists})')
print(f'\n資料庫統計：')
for src, cnt in rows:
    label = 'shadcn/ui  ' if src == 'shadcn' else 'TailwindCSS'
    print(f'  {label}：{cnt} 塊')
print(f'  合計         ：{total_rows} 塊')

索引 shadcn/ui 文件
[01/40] ✓ 已存在：docs
[02/40] ✓ 已存在：installation
[03/40] ✓ 已存在：cli
[04/40] ✓ 已存在：theming
[05/40] ✓ 已存在：dark-mode
[06/40] ✓ 已存在：accordion
[07/40] ✓ 已存在：alert
[08/40] ✓ 已存在：alert-dialog
[09/40] ✓ 已存在：avatar
[10/40] ✓ 已存在：badge
[11/40] ✓ 已存在：button
[12/40] ✓ 已存在：calendar
[13/40] ✓ 已存在：card
[14/40] ✓ 已存在：checkbox
[15/40] ✓ 已存在：command
[16/40] ✓ 已存在：context-menu
[17/40] ✓ 已存在：data-table
[18/40] ✓ 已存在：date-picker
[19/40] ✓ 已存在：dialog
[20/40] ✓ 已存在：dropdown-menu
[21/40] ✓ 已存在：form
[22/40] ✓ 已存在：hover-card
[23/40] ✓ 已存在：input
[24/40] ✓ 已存在：label
[25/40] ✓ 已存在：navigation-menu
[26/40] ✓ 已存在：popover
[27/40] ✓ 已存在：progress
[28/40] ✓ 已存在：radio-group
[29/40] ✓ 已存在：select
[30/40] ✓ 已存在：separator
[31/40] ✓ 已存在：sheet
[32/40] ✓ 已存在：skeleton
[33/40] ✓ 已存在：slider
[34/40] ✓ 已存在：switch
[35/40] ✓ 已存在：table
[36/40] ✓ 已存在：tabs
[37/40] ✓ 已存在：textarea
[38/40] ✓ 已存在：toast
[39/40] ✓ 已存在：toggle
[40/40] ✓ 已存在：tooltip

索引 TailwindCSS 文件
[01/46] ✓ 已存在：installation
[02/46] ✓ 已存在：utility-first
[03/46] ✓ 已存在

In [94]:
# Cell 9: 語義搜尋函式

def semantic_search(
    query: str,
    source_filter: str = 'both',
    top_k: int = 5
) -> List[dict]:
    """以向量相似度搜尋最相關的文件段落。"""
    q_emb = get_embedding(query, is_query=True)

    conn = get_conn()
    with conn.cursor() as cur:
        if source_filter in ('shadcn', 'tailwind'):
            cur.execute(
                '''
                SELECT source, url, title, content,
                       1 - (embedding <=> %s::vector) AS sim
                FROM docs
                WHERE source = %s
                ORDER BY embedding <=> %s::vector
                LIMIT %s
                ''',
                (q_emb, source_filter, q_emb, top_k)
            )
        else:
            cur.execute(
                '''
                SELECT source, url, title, content,
                       1 - (embedding <=> %s::vector) AS sim
                FROM docs
                ORDER BY embedding <=> %s::vector
                LIMIT %s
                ''',
                (q_emb, q_emb, top_k)
            )

        results = [
            {
                'source':     row[0],
                'url':        row[1],
                'title':      row[2],
                'content':    row[3],
                'similarity': float(row[4]),
            }
            for row in cur.fetchall()
        ]
    conn.close()
    return results


# ── 快速測試 ──────────────────────────────────────────────────
print('🔍 測試搜尋："how to add a button component"')
test_hits = semantic_search('how to add a button component', top_k=3)
for r in test_hits:
    label = 'shadcn/ui' if r['source'] == 'shadcn' else 'TailwindCSS'
    print(f'  [{label}] {r["title"]}  sim={r["similarity"]:.3f}')

print()
print('🔍 測試搜尋："flex justify-center items-center"')
test_hits2 = semantic_search('flex justify-center items-center', top_k=3)
for r in test_hits2:
    label = 'shadcn/ui' if r['source'] == 'shadcn' else 'TailwindCSS'
    print(f'  [{label}] {r["title"]}  sim={r["similarity"]:.3f}')

🔍 測試搜尋："how to add a button component"


  [shadcn/ui] Dark Mode  sim=0.863
  [shadcn/ui] Accordion  sim=0.863
  [shadcn/ui] Alert  sim=0.863

🔍 測試搜尋："flex justify-center items-center"
  [TailwindCSS] justify-content  sim=0.881
  [TailwindCSS] align-items  sim=0.865
  [TailwindCSS] display  sim=0.832


In [ ]:
# Cell 10: 初始化 Gemini 並定義 RAG 問答函式

genai.configure(api_key=GEMINI_API_KEY)
gemini = genai.GenerativeModel(
    model_name=GEMINI_MODEL,
    generation_config=genai.GenerationConfig(
        temperature=0.3,
        max_output_tokens=1024,
    ),
    system_instruction=(
        'You are a helpful assistant specializing in shadcn/ui and TailwindCSS. '
        'Answer accurately using the provided documentation context. '
        'Include short code examples when relevant. '
        'If the context does not contain the answer, say so clearly.'
    ),
)
print(f'✅ Gemini ({GEMINI_MODEL}) 就緒')


def ask(
    question: str,
    source_filter: str = 'both',
    top_k: int = 5,
) -> Tuple[str, List[dict]]:
    """RAG 問答：搜尋相關段落後，交由 Gemini API 生成回答。"""
    hits = semantic_search(question, source_filter=source_filter, top_k=top_k)

    context_blocks = []
    for h in hits:
        label = 'shadcn/ui' if h['source'] == 'shadcn' else 'TailwindCSS'
        context_blocks.append('[' + label + ' — ' + h['title'] + ']\n' + h['content'])
    context = '\n\n---\n\n'.join(context_blocks)

    prompt = (
        '=== Documentation Context ===\n'
        + context
        + '\n=============================\n\nQuestion: '
        + question
    )

    # Retry with exponential backoff on rate limit (429)
    import google.api_core.exceptions
    delay = 10
    for attempt in range(5):
        try:
            response = gemini.generate_content(prompt)
            return response.text.strip(), hits
        except google.api_core.exceptions.ResourceExhausted as e:
            if attempt == 4:
                raise
            print(f'  ⏳ Rate limit hit, waiting {delay}s... (attempt {attempt + 1}/5)')
            time.sleep(delay)
            delay *= 2


# ── 測試 5 題 ────────────────────────────────────────────────
test_questions = [
    ('How do I install the Button component in shadcn/ui?', 'shadcn'),
    ('What variants does the Badge component support?', 'shadcn'),
    ('How do I use Tailwind to center a div with flexbox?', 'tailwind'),
    ('What are the responsive breakpoint prefixes in Tailwind?', 'tailwind'),
    ('How do I implement dark mode with shadcn/ui and Tailwind?', 'both'),
]

print('=' * 60)
print('問答測試 (5 題)')
print('=' * 60)
for idx, (q, src) in enumerate(test_questions, 1):
    print(f'Q{idx} [{src}]: {q}')
    answer, _ = ask(q, source_filter=src)
    print(f'A{idx}:', answer[:400], '...' if len(answer) > 400 else '')
    print('-' * 60)
    if idx < len(test_questions):
        time.sleep(5)  # avoid rate limit between test calls


✅ Gemini (gemini-3-flash-preview) 就緒
問答測試 (5 題)
Q1 [shadcn]: How do I install the Button component in shadcn/ui?


A1: The provided documentation context does not contain information on how to install the Button component.

However, based on standard **shadcn/ui** usage, you can install the Button component by running the following command in your terminal:

```bash
npx shadcn-ui@latest add button
```

This command will automatically create a `button.tsx` file in your `components/ui` directory and install any nece ...
------------------------------------------------------------
Q2 [shadcn]: What variants does the Badge component support?
A2: The provided documentation context does not contain the specific page or API reference for the **Badge** component. While "Badge" is listed in the components navigation menu, the detailed documentation for its variants is not included in the text provided. 
------------------------------------------------------------
Q3 [tailwind]: How do I use Tailwind to center a div with flexbox?
A3: To center a `div` using Flexbox in Tailwind CSS, you combine the `flex` uti

In [ ]:
# Cell 11: Gradio Chat UI

def respond_gradio(
    message: str,
    history: list,
    source_filter: str,
) -> Tuple[str, str]:
    """Gradio 回呼：回傳 (answer, context_display)"""
    answer, hits = ask(message, source_filter=source_filter)

    ctx_parts = []
    for h in hits:
        label = 'shadcn/ui' if h['source'] == 'shadcn' else 'TailwindCSS'
        ctx_parts.append(
            '[' + label + '] ' + h['title'] + '\n'
            + '   Similarity : ' + f"{h['similarity']:.3f}" + '\n'
            + '   URL        : ' + h['url'] + '\n'
            + '   ' + h['content'][:280] + '...'
        )
    ctx_str = '\n\n'.join(ctx_parts)
    return answer, ctx_str


# ── 建立 Gradio Blocks UI ─────────────────────────────────────
with gr.Blocks(
    title='shadcn/ui & TailwindCSS Knowledge Base',
    theme=gr.themes.Soft(),
    css='.gradio-container { max-width: 1280px !important }',
) as demo:

    gr.Markdown("""
    # shadcn/ui & TailwindCSS Knowledge Base
    使用 RAG（檢索增強生成）回答關於 shadcn/ui 元件與 TailwindCSS 工具的問題。
    """)

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label='對話',
                height=520,
            )
            with gr.Row():
                msg_box = gr.Textbox(
                    placeholder='例如：How do I use the Dialog component in shadcn/ui?',
                    label='',
                    scale=4,
                    lines=1,
                )
                source_dd = gr.Dropdown(
                    choices=['both', 'shadcn', 'tailwind'],
                    value='both',
                    label='語料來源',
                    scale=1,
                )
            with gr.Row():
                send_btn  = gr.Button('Send', variant='primary', scale=2)
                clear_btn = gr.Button('Clear', scale=1)

            gr.Examples(
                label='範例問題',
                examples=[
                    ['How do I install and use the Button component?', 'shadcn'],
                    ['What variants does the Badge component support?', 'shadcn'],
                    ['How does the Dialog (modal) component work in shadcn/ui?', 'shadcn'],
                    ['How do I center elements with Tailwind flexbox?', 'tailwind'],
                    ['What are the responsive breakpoints in Tailwind (sm, md, lg)?', 'tailwind'],
                    ['How do I implement dark mode with shadcn/ui and Tailwind?', 'both'],
                ],
                inputs=[msg_box, source_dd],
            )

        with gr.Column(scale=1, min_width=320):
            context_box = gr.Textbox(
                label='Retrieved Chunks',
                placeholder='問問題後，這裡會顯示向量搜尋取回的段落...',
                lines=30,
                interactive=False,
                max_lines=30,
            )

    # ── 事件綁定 ──────────────────────────────────────────────
    def user_turn(msg, history):
        history = history + [{'role': 'user', 'content': msg}]
        return '', history

    def bot_turn(history, source_filter):
        question = history[-1]['content']
        answer, ctx_str = respond_gradio(question, history[:-1], source_filter)
        history = history + [{'role': 'assistant', 'content': answer}]
        return history, ctx_str

    submit_inputs  = [msg_box, chatbot]
    submit_outputs = [msg_box, chatbot]

    msg_box.submit(
        user_turn, submit_inputs, submit_outputs
    ).then(
        bot_turn, [chatbot, source_dd], [chatbot, context_box]
    )

    send_btn.click(
        user_turn, submit_inputs, submit_outputs
    ).then(
        bot_turn, [chatbot, source_dd], [chatbot, context_box]
    )

    clear_btn.click(lambda: ([], ''), outputs=[chatbot, context_box])


print('🚀 啟動 Gradio UI...')
demo.launch(share=False, server_port=7860)


/tmp/ipykernel_66734/1768082376.py:26: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


🚀 啟動 Gradio UI...
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/home/max/workspace/nycu_gay_2026/.venv/lib/python3.14/site-packages/gradio/queueing.py", line 785, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/home/max/workspace/nycu_gay_2026/.venv/lib/python3.14/site-packages/gradio/route_utils.py", line 358, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "/home/max/workspace/nycu_gay_2026/.venv/lib/python3.14/site-packages/gradio/blocks.py", line 2174, in process_api
    data = await self.postprocess_data(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        block_fn, result["prediction"], state
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/home/max/workspace/nycu_gay_2026/.venv/lib/python3.14/site-packages/gradio/blocks.py", line 1946, in postprocess_data
    predic